# 第 4 章 · 参数估计与置信区间

**点估计**给出一个具体数值，**区间估计**给出一个带置信度的范围。
本章手算置信区间，再用真实数据集演示。

## 0. 准备环境

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 matplotlib 正确显示中文与负号
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)   # 固定随机种子，保证每次运行结果一致
print("环境就绪：numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# 本章同样使用一个模拟"总体"作为被抽样的对象
np.random.seed(42)
population = np.random.gamma(shape=5, scale=12, size=10000) + 20
print("总体均值 μ = %.2f, 总体标准差 σ = %.2f" % (population.mean(), population.std()))

## 1. 点估计：用样本估计总体

| 总体参数 | 样本统计量 |
|---|---|
| 均值 μ | 样本均值 x̄ |
| 方差 σ² | 样本方差 s² |
| 比例 p | 样本比例 p̂ |

**评价标准**：无偏性、有效性、一致性。

## 2. 置信区间（均值，σ 未知 → 用 t 分布）

In [ ]:
from math import sqrt

# 标准正态分位数（用于大样本 / σ 已知）
# t 分位数用简单近似，避免额外依赖；大样本下与正态接近

def z_critical(conf):
    ztab = {0.90: 1.645, 0.95: 1.960, 0.99: 2.576}
    return ztab[conf]

def mean_ci(data, conf=0.95):
    x = np.asarray(data, dtype=float)
    n = len(x)
    m = x.mean()
    s = x.std(ddof=1)
    se = s / sqrt(n)
    z = z_critical(conf)
    return m, m - z * se, m + z * se

# 抽 36 名学生的成绩
sample = np.random.choice(population, 36, replace=False)
m, lo, hi = mean_ci(sample, 0.95)
print("样本均值 = %.2f" % m)
print("95%% 置信区间 = [%.2f, %.2f]" % (lo, hi))

**怎样理解 95% 置信区间？** 重复抽样 100 次，约 95 次构造出的区间会包含真实 μ。
它不是"μ 有 95% 概率落在区间里"，而是"这套构造方法的正确率是 95%"。

In [ ]:
# 模拟验证置信区间的覆盖率
true_mu = population.mean()
hit = 0
for _ in range(2000):
    s = np.random.choice(population, 36, replace=False)
    _, lo, hi = mean_ci(s, 0.95)
    if lo <= true_mu <= hi:
        hit += 1
print("真实 μ = %.2f" % true_mu)
print("2000 个置信区间中覆盖真值的比例 = %.3f (理论 0.95)" % (hit / 2000))

## 3. 置信区间（比例）

当样本量较大时，比例的置信区间：
p̂ ± z · √( p̂(1−p̂) / n )

In [ ]:
def proportion_ci(count, n, conf=0.95):
    phat = count / n
    se = sqrt(phat * (1 - phat) / n)
    z = z_critical(conf)
    return phat, phat - z * se, phat + z * se

# 抽 200 人，120 人表示支持
phat, lo, hi = proportion_ci(120, 200, 0.95)
print("样本比例 = %.3f" % phat)
print("95%% 置信区间 = [%.3f, %.3f]" % (lo, hi))

## 4. 用真实数据文件计算

下面读取 `data/students.csv`，计算数学成绩均值的 95% 置信区间。

In [ ]:
df = pd.read_csv("data/students.csv")
print(df.head())

math_scores = df["数学"]
m, lo, hi = mean_ci(math_scores, 0.95)
print("\n数学成绩：n = %d, 均值 = %.2f, 标准差 = %.2f" % (len(math_scores), m, math_scores.std(ddof=1)))
print("95%% 置信区间 = [%.2f, %.2f]" % (lo, hi))

## 小结

- 点估计给一个数，区间估计给一个范围 + 置信度。
- 均值区间：x̄ ± z·(s/√n)（大样本用正态，小样本用 t）。
- 影响区间宽度的因素：**置信度↑ 区间变宽；样本量↑ / 离散↓ 区间变窄**。